# Plot in vitro trajectories

In [9]:
here::i_am("revisions/01_invivo_lineages/01_manual_trajectory.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(edgeR))
suppressPackageStartupMessages(library(slingshot))



BPPARAM <- BiocParallel::bpparam()
BPPARAM$workers = 16

# Multi core using future - built in to seurat
plan("multicore", workers = 16)
options(future.globals.maxSize = 50 * 1024 ^ 3) # for 50 Gb RAM

set.seed(1234)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code



In [6]:
args = list()

# Metadata
args$metadata = file.path(io$basedir, 'results/rna_atac/clustering/metadata_celltype_annotated_v2.txt.gz')

# RNA_sce
args$rna_sce = io$rna.atlas.sce

# outdir
args$outdir = file.path(io$basedir, 'results/revisions/01_invivo_lineages/')
dir.create(args$outdir, recursive=TRUE, showWarnings =FALSE)

In [7]:
# Load meta
meta = fread(args$metadata)[day %in% c('D3', 'D3.5', 'D4', 'D4.5', 'D5')] %>% 
    .[celltype_v2 %in% c('Primitive_Streak', 'Early_Mes_EOi','Early_Mes_EOd','Posterior_Mes','HE_Precursor', 'HE', 'Allantois_Precursor')]

In [8]:
vitro_rna.sce = readRDS(file.path(io$basedir, 'results/rna_atac/trajectory/v3', 'rna_sce.rds'))

In [10]:
# Assign cells to lineages
set.seed(1234)
assign_lineages = tradeSeq:::.assignCells(assays(vitro_rna.sce$slingshot)$weights)
#colnames(assign_lineages) = c('Trajectory1', 'Trajectory2')
pseudotime = slingPseudotime(vitro_rna.sce$slingshot, na=T)

Traj_assignment = as.data.table(assign_lineages, keep.rownames=T) %>%
    setnames(c('cell', paste0('Trajectory', 1:(length(.)-1))))

In [28]:
meta_plot = cbind(merge(meta, Traj_assignment), pseudotime[meta$cell,]) %>%
    .[,pseudotime := ifelse(Trajectory1 == 1, Lineage1, 
                            ifelse(Trajectory2 == 1, Lineage2, NA))]

In [29]:
meta_atlas = fread(io$rna.atlas.metadata)

In [33]:
meta_plot = meta_plot %>% 
    cbind(., meta_atlas[match(meta_plot$closest.cell_mnn, cell)][,.(umapX, umapY)])

In [38]:
palette1 <- colorRampPalette(c("darkblue", "orange", "chartreuse", 'chartreuse4')) # YS trajectory palette
palette2 <- colorRampPalette(c("darkblue", "orange", "orchid1", 'darkorchid4'))  # Alnt trajectory palette

t1 = ggplot(meta_plot[Trajectory1 == 1], aes(umapX, umapY, color = pseudotime)) + 
    ggrastr::rasterise(geom_point(data = meta_atlas, 
               color='black', size = -0.05), dpi = 600) + 
    ggrastr::rasterise(geom_point(data = meta_atlas, 
               color='grey90', size = -0.15), dpi = 600) + 
    geom_point(size = -0) + 
    #viridis::scale_color_viridis() +
    scale_color_gradientn(colors = c("darkblue", "orange", "chartreuse", 'chartreuse4'),
                         guide = guide_colorbar(
                            barwidth = 3, barheight = 0.5,
                            frame.colour = "black",
                            ticks.colour = 'black')) + 
    coord_fixed(ratio = 0.75) + 
    theme_void() + 
    theme(legend.text = element_text(size = 6),
          legend.title = element_text(size = 7),
          legend.position = 'bottom')

t2 = ggplot(meta_plot[Trajectory2 == 1], aes(umapX, umapY, color = pseudotime)) + 
    ggrastr::rasterise(geom_point(data = meta_atlas, 
               color='black', size = -0.05), dpi = 600) + 
    ggrastr::rasterise(geom_point(data = meta_atlas, 
               color='grey90', size = -0.15), dpi = 600) +     
    geom_point(size = -0) + 
    #viridis::scale_color_viridis() +
    scale_color_gradientn(colors = c("darkblue", "orange", "orchid1", 'darkorchid4'),
                            guide = guide_colorbar(
                            barwidth = 3, barheight = 0.5,
                            frame.colour = "black",
                            ticks.colour = 'black')) +   
    coord_fixed(ratio = 0.75) + 
    theme_void() + 
    theme(legend.text = element_text(size = 6),
          legend.title = element_text(size = 7),
          legend.position = 'bottom')

options(repr.plot.width=10, repr.plot.height=5)
#ggarrange(t1, t2)

In [39]:
ggsave(file.path(args$outdir, 'invitro_umap_pseudotime.pdf'), 
       plot = ggarrange(t1, t2), 
       width = 120, height = 60, units = "mm")